# Module 1: DistilBERT Fine-tuning for NLC Classification

## WanderWise+ - Intelligent Tourism Route Planning

This notebook fine-tunes a pre-trained DistilBERT model for multi-label text classification.

### Categories (7 labels):
1. **beaches** - Coastal areas, water activities
2. **historical** - Forts, churches, temples, monuments
3. **adventure** - Water sports, trekking, extreme activities
4. **nature** - Wildlife, waterfalls, plantations
5. **food** - Restaurants, local cuisine, food tours
6. **nightlife** - Clubs, bars, casinos, parties
7. **shopping** - Markets, handicrafts, souvenirs

---

## 1. Setup and Installation

### Required Libraries:
- `transformers` - Hugging Face transformers for pre-trained models
- `torch` - PyTorch for deep learning
- `datasets` - Hugging Face datasets library
- `scikit-learn` - For metrics and evaluation
- `pandas`, `numpy` - Data manipulation

In [ ]:
# ============================================================
# INSTALLATION - Run this cell first if libraries are not installed
# ============================================================
# Uncomment and run if needed:

# !pip install transformers torch datasets scikit-learn pandas numpy matplotlib seaborn accelerate evaluate

In [ ]:
# ============================================================
# IMPORTS - All required libraries
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np

# PyTorch for deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# Hugging Face transformers
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# Scikit-learn for metrics and data splitting
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# System utilities
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

---
## 2. Configuration

All hyperparameters and settings are defined here for easy modification.

In [ ]:
# ============================================================
# CONFIGURATION - All hyperparameters in one place
# ============================================================

# ----- Model Configuration -----
MODEL_NAME = 'distilbert-base-uncased'  # Pre-trained model from Hugging Face
MAX_LENGTH = 128                         # Maximum sequence length for tokenizer
NUM_LABELS = 7                           # Number of output categories

# ----- Training Configuration -----
BATCH_SIZE = 16                          # Batch size for training (reduce if OOM)
EPOCHS = 5                               # Number of training epochs
LEARNING_RATE = 2e-5                     # Learning rate (typical for fine-tuning)
WEIGHT_DECAY = 0.01                      # Weight decay for regularization
WARMUP_RATIO = 0.1                       # Warmup steps ratio
EARLY_STOPPING_PATIENCE = 2              # Stop if no improvement for N epochs

# ----- Data Configuration -----
TEST_SIZE = 0.2                          # 20% for testing
VAL_SIZE = 0.1                           # 10% of training for validation
DATA_PATH = 'nlc_training_data_5000.csv' # Path to your training data

# ----- Output Configuration -----
OUTPUT_DIR = 'backend/app/models/nlc/distilbert_finetuned/'
SAVE_MODEL = True                        # Whether to save the fine-tuned model

# ----- Category Labels -----
CATEGORIES = ['beaches', 'historical', 'adventure', 'nature', 'food', 'nightlife', 'shopping']
LABEL2ID = {label: i for i, label in enumerate(CATEGORIES)}
ID2LABEL = {i: label for i, label in enumerate(CATEGORIES)}

# Print configuration summary
print("=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Categories: {CATEGORIES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max sequence length: {MAX_LENGTH}")
print(f"Output directory: {OUTPUT_DIR}")
print("=" * 60)

---
## 3. Data Loading and Preprocessing

Load the CSV data and prepare it for the model.

In [ ]:
# ============================================================
# LOAD DATA - Read CSV file with training data
# ============================================================

# Load the dataset
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# ============================================================
# DATA EXPLORATION - Understand the dataset distribution
# ============================================================

# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

# Count samples per category
print("Samples per category:")
category_counts = df[CATEGORIES].sum()
for cat, count in category_counts.items():
    print(f"  {cat}: {count} ({count/len(df)*100:.1f}%)")
print()

# Count labels per sample (multi-label distribution)
labels_per_sample = df[CATEGORIES].sum(axis=1)
print("Labels per sample distribution:")
print(labels_per_sample.value_counts().sort_index())

In [ ]:
# ============================================================
# VISUALIZE DATA DISTRIBUTION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Category distribution
axes[0].bar(CATEGORIES, category_counts.values, color='steelblue')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Number of Samples')
axes[0].set_title('Samples per Category')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Labels per sample
labels_per_sample.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_xlabel('Number of Labels')
axes[1].set_ylabel('Number of Samples')
axes[1].set_title('Multi-label Distribution')

plt.tight_layout()
plt.savefig('nlc_distilbert_data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved as 'nlc_distilbert_data_distribution.png'")

In [ ]:
# ============================================================
# PREPARE LABELS - Convert dataframe columns to numpy arrays
# ============================================================

# Extract texts and labels
texts = df['text'].values
labels = df[CATEGORIES].values.astype(np.float32)  # Convert to float for PyTorch

print(f"Texts shape: {texts.shape}")
print(f"Labels shape: {labels.shape}")
print(f"\nExample text: {texts[0]}")
print(f"Example labels: {labels[0]}")
print(f"Categories: {[CATEGORIES[i] for i, val in enumerate(labels[0]) if val == 1]}")

In [ ]:
# ============================================================
# TRAIN/VAL/TEST SPLIT - Split data into training, validation, and test sets
# ============================================================

# First split: separate test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    texts, labels, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_SEED
)

# Second split: separate validation set from remaining data
# val_size is relative to temp data, so we calculate the ratio
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=val_ratio, 
    random_state=RANDOM_SEED
)

print("Data split summary:")
print(f"  Training samples: {len(X_train)} ({len(X_train)/len(texts)*100:.1f}%)")
print(f"  Validation samples: {len(X_val)} ({len(X_val)/len(texts)*100:.1f}%)")
print(f"  Test samples: {len(X_test)} ({len(X_test)/len(texts)*100:.1f}%)")

---
## 4. Dataset and Tokenization

Create a custom PyTorch Dataset class and tokenize the text data.

In [ ]:
# ============================================================
# INITIALIZE TOKENIZER - Load the DistilBERT tokenizer
# ============================================================

# Load pre-trained tokenizer
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Model max length: {tokenizer.model_max_length}")

In [ ]:
# ============================================================
# CUSTOM DATASET CLASS - PyTorch Dataset for our NLC data
# ============================================================

class NLCDataset(Dataset):
    """
    Custom PyTorch Dataset for multi-label text classification.
    
    Args:
        texts: Array of text strings
        labels: Array of multi-label binary vectors
        tokenizer: Hugging Face tokenizer
        max_length: Maximum sequence length
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        """Return the number of samples."""
        return len(self.texts)
    
    def __getitem__(self, idx):
        """
        Get a single sample.
        
        Returns:
            dict with keys: input_ids, attention_mask, labels
        """
        text = str(self.texts[idx])
        labels = self.labels[idx]
        
        # Tokenize the text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,      # Add [CLS] and [SEP] tokens
            max_length=self.max_length,   # Pad/truncate to max length
            padding='max_length',         # Pad to max_length
            truncation=True,              # Truncate if longer than max_length
            return_tensors='pt',          # Return PyTorch tensors
            return_attention_mask=True    # Return attention mask
        )
        
        # Remove the batch dimension added by tokenizer
        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.FloatTensor(labels)  # Float for multi-label classification
        }

# Test the dataset class
print("Creating dataset objects...")
train_dataset = NLCDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = NLCDataset(X_val, y_val, tokenizer, MAX_LENGTH)
test_dataset = NLCDataset(X_test, y_test, tokenizer, MAX_LENGTH)

print(f"\nDatasets created:")
print(f"  Training: {len(train_dataset)} samples")
print(f"  Validation: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")

# Check a sample
sample = train_dataset[0]
print(f"\nSample from training set:")
print(f"  Input IDs shape: {sample['input_ids'].shape}")
print(f"  Attention mask shape: {sample['attention_mask'].shape}")
print(f"  Labels shape: {sample['labels'].shape}")
print(f"  Labels: {sample['labels']}")

---
## 5. Model Initialization

Load the pre-trained DistilBERT model and configure it for multi-label classification.

In [ ]:
# ============================================================
# INITIALIZE MODEL - Load pre-trained DistilBERT for sequence classification
# ============================================================

# Load pre-trained model with classification head
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",  # Important for multi-label!
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Print model architecture summary
print("Model loaded successfully!")
print(f"\nModel architecture:")
print(f"  Base model: DistilBERT")
print(f"  Number of labels: {NUM_LABELS}")
print(f"  Problem type: multi_label_classification")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ============================================================
# DEVICE SETUP - Move model to GPU if available
# ============================================================

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Move model to device
model = model.to(device)
print(f"\nModel moved to {device}")

---
## 6. Metrics and Evaluation Functions

Define custom metrics for multi-label classification evaluation.

In [ ]:
# ============================================================
# METRICS FUNCTIONS - Custom metrics for multi-label classification
# ============================================================

def compute_metrics_multilabel(eval_pred, threshold=0.5):
    """
    Compute metrics for multi-label classification.
    
    Args:
        eval_pred: Tuple of (predictions, labels)
        threshold: Classification threshold for sigmoid outputs
    
    Returns:
        dict: Dictionary of metric names and values
    """
    predictions, labels = eval_pred
    
    # Apply sigmoid to get probabilities
    probs = torch.sigmoid(torch.tensor(predictions))
    
    # Convert probabilities to binary predictions
    preds = (probs > threshold).numpy().astype(int)
    
    # Calculate metrics
    # Note: Use samples average for multi-label (calculates metrics for each instance)
    f1_micro = f1_score(labels, preds, average='micro', zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    f1_samples = f1_score(labels, preds, average='samples', zero_division=0)
    
    precision_micro = precision_score(labels, preds, average='micro', zero_division=0)
    recall_micro = recall_score(labels, preds, average='micro', zero_division=0)
    
    hamming = hamming_loss(labels, preds)
    
    # Subset accuracy (exact match)
    exact_match = accuracy_score(labels, preds)
    
    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_samples': f1_samples,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro,
        'hamming_loss': hamming,
        'exact_match': exact_match
    }

print("Metrics functions defined!")
print("\nMetrics that will be tracked:")
print("  - f1_micro: F1 score calculated globally")
print("  - f1_macro: F1 score averaged per class")
print("  - f1_samples: F1 score averaged per sample")
print("  - precision_micro: Precision calculated globally")
print("  - recall_micro: Recall calculated globally")
print("  - hamming_loss: Fraction of wrong labels")
print("  - exact_match: Percentage of exact label matches")

---
## 7. Training Configuration

Set up training arguments and the Trainer object.

In [ ]:
# ============================================================
# TRAINING ARGUMENTS - Configure training parameters
# ============================================================

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',              # Directory for checkpoints and logs
    num_train_epochs=EPOCHS,             # Total number of training epochs
    per_device_train_batch_size=BATCH_SIZE,  # Batch size per device
    per_device_eval_batch_size=BATCH_SIZE,   # Batch size for evaluation
    learning_rate=LEARNING_RATE,         # Learning rate
    weight_decay=WEIGHT_DECAY,           # Weight decay for regularization
    warmup_ratio=WARMUP_RATIO,           # Warmup steps ratio
    
    # Evaluation settings
    eval_strategy='epoch',               # Evaluate at end of each epoch
    save_strategy='epoch',               # Save at end of each epoch
    load_best_model_at_end=True,         # Load best model at end of training
    metric_for_best_model='f1_micro',    # Metric to use for best model
    greater_is_better=True,              # Higher F1 is better
    
    # Logging settings
    logging_dir='./logs',                # Directory for logs
    logging_steps=50,                    # Log every N steps
    
    # Other settings
    seed=RANDOM_SEED,                    # Random seed for reproducibility
    fp16=torch.cuda.is_available(),      # Use mixed precision if GPU available
    gradient_accumulation_steps=1,       # Gradient accumulation (increase if OOM)
    report_to='none',                    # Disable external logging (wandb, etc.)
)

print("Training arguments configured!")
print(f"\nKey settings:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Mixed precision (FP16): {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# INITIALIZE TRAINER - Create the Hugging Face Trainer
# ============================================================

# Create Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_multilabel,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)]
)

print("Trainer initialized!")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE} epochs")

---
## 8. Training

Start the fine-tuning process. This may take 15-30 minutes on GPU or 2-3 hours on CPU.

In [ ]:
# ============================================================
# START TRAINING - Fine-tune the model
# ============================================================

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Device: {device}")
print("=" * 60)
print()

# Start training
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETED!")
print("=" * 60)

In [ ]:
# ============================================================
# TRAINING SUMMARY - Display training results
# ============================================================

# Print training metrics
print("Training Summary:")
print("-" * 40)
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

# Save training metrics
trainer.save_metrics('train', train_result.metrics)

---
## 9. Evaluation on Test Set

Evaluate the fine-tuned model on the held-out test set.

In [ ]:
# ============================================================
# TEST SET EVALUATION - Evaluate on unseen test data
# ============================================================

print("Evaluating on test set...")
print("=" * 60)

# Evaluate on test set
test_results = trainer.evaluate(test_dataset)

print("\nTest Set Results:")
print("-" * 40)
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save test metrics
trainer.save_metrics('test', test_results)

In [ ]:
# ============================================================
# DETAILED PREDICTIONS - Get predictions for detailed analysis
# ============================================================

# Get predictions on test set
predictions = trainer.predict(test_dataset)

# Apply sigmoid and threshold
probs = torch.sigmoid(torch.tensor(predictions.predictions))
preds = (probs > 0.5).numpy().astype(int)
labels = predictions.label_ids.astype(int)

print(f"Predictions shape: {preds.shape}")
print(f"Labels shape: {labels.shape}")

In [ ]:
# ============================================================
# CLASSIFICATION REPORT - Detailed per-class metrics
# ============================================================

print("Classification Report (per category):")
print("=" * 60)
print(classification_report(labels, preds, target_names=CATEGORIES, zero_division=0))

In [ ]:
# ============================================================
# CONFUSION MATRICES - Visualize per-class performance
# ============================================================

# Calculate confusion matrices for each category
conf_matrices = multilabel_confusion_matrix(labels, preds)

# Plot confusion matrices
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (cm, category) in enumerate(zip(conf_matrices, CATEGORIES)):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    axes[i].set_title(f'{category}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

# Hide the last subplot (8 subplots for 7 categories)
axes[7].axis('off')

plt.suptitle('Confusion Matrices per Category', fontsize=14)
plt.tight_layout()
plt.savefig('nlc_distilbert_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nConfusion matrices saved as 'nlc_distilbert_confusion_matrices.png'")

In [ ]:
# ============================================================
# F1 SCORES VISUALIZATION - Bar chart of F1 scores per category
# ============================================================

# Calculate F1 scores per category
f1_per_class = f1_score(labels, preds, average=None, zero_division=0)

# Create bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(CATEGORIES, f1_per_class, color='steelblue')

# Add value labels on bars
for bar, score in zip(bars, f1_per_class):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.3f}', ha='center', va='bottom', fontsize=10)

plt.xlabel('Category')
plt.ylabel('F1 Score')
plt.title('F1 Score per Category (Test Set)')
plt.ylim(0, 1.1)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('nlc_distilbert_f1_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nF1 scores chart saved as 'nlc_distilbert_f1_scores.png'")

---
## 10. Save the Fine-tuned Model

Save the model and tokenizer for later use in production.

In [ ]:
# ============================================================
# SAVE MODEL - Save the fine-tuned model and tokenizer
# ============================================================

if SAVE_MODEL:
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Save the model
    model.save_pretrained(OUTPUT_DIR)
    print(f"Model saved to: {OUTPUT_DIR}")
    
    # Save the tokenizer
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Tokenizer saved to: {OUTPUT_DIR}")
    
    # Save metadata
    metadata = {
        'model_name': MODEL_NAME,
        'num_labels': NUM_LABELS,
        'categories': CATEGORIES,
        'max_length': MAX_LENGTH,
        'training_date': datetime.now().isoformat(),
        'training_samples': len(train_dataset),
        'test_f1_micro': float(test_results.get('eval_f1_micro', 0)),
        'test_f1_macro': float(test_results.get('eval_f1_macro', 0)),
        'test_exact_match': float(test_results.get('eval_exact_match', 0)),
        'hyperparameters': {
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'max_length': MAX_LENGTH
        }
    }
    
    with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"Metadata saved to: {OUTPUT_DIR}metadata.json")
    
    print("\n" + "=" * 60)
    print("MODEL SAVED SUCCESSFULLY!")
    print("=" * 60)
    print(f"\nFiles saved:")
    print(f"  - config.json")
    print(f"  - pytorch_model.bin")
    print(f"  - tokenizer_config.json")
    print(f"  - vocab.txt")
    print(f"  - metadata.json")
else:
    print("Model saving skipped (SAVE_MODEL=False)")

---
## 11. Inference Example

Demonstrate how to load and use the saved model for predictions.

In [ ]:
# ============================================================
# LOAD SAVED MODEL - Example of loading the fine-tuned model
# ============================================================

# Load model and tokenizer from saved directory
loaded_model = DistilBertForSequenceClassification.from_pretrained(OUTPUT_DIR)
loaded_tokenizer = DistilBertTokenizer.from_pretrained(OUTPUT_DIR)

# Move to device
loaded_model = loaded_model.to(device)
loaded_model.eval()  # Set to evaluation mode

print("Model and tokenizer loaded successfully!")

In [ ]:
# ============================================================
# PREDICTION FUNCTION - Function to make predictions on new text
# ============================================================

def predict_categories(text, model, tokenizer, categories=CATEGORIES, threshold=0.5):
    """
    Predict categories for a given text.
    
    Args:
        text: Input text string
        model: Fine-tuned model
        tokenizer: Tokenizer
        categories: List of category names
        threshold: Classification threshold
    
    Returns:
        dict: Dictionary with categories and their probabilities
    """
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True
    )
    
    # Move inputs to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)
    
    # Convert to numpy
    probs = probs.cpu().numpy()[0]
    
    # Get predicted categories
    predicted = [(cat, float(prob)) for cat, prob in zip(categories, probs) if prob > threshold]
    
    # Sort by probability
    predicted = sorted(predicted, key=lambda x: x[1], reverse=True)
    
    return {
        'text': text,
        'predicted_categories': [cat for cat, _ in predicted],
        'probabilities': {cat: float(prob) for cat, prob in zip(categories, probs)},
        'all_predictions': predicted
    }

print("Prediction function defined!")

In [ ]:
# ============================================================
# TEST PREDICTIONS - Test the model with sample inputs
# ============================================================

# Sample texts to test
test_texts = [
    "I want to visit beaches and try water sports",
    "Looking for historical forts and local food",
    "Where can I go for nightlife and shopping?",
    "I love nature and wildlife",
    "Want to explore temples and churches"
]

print("Sample Predictions:")
print("=" * 60)

for text in test_texts:
    result = predict_categories(text, loaded_model, loaded_tokenizer)
    print(f"\nText: {result['text']}")
    print(f"Predicted: {result['predicted_categories']}")
    print(f"Probabilities: {result['probabilities']}")
    print("-" * 40)

---
## 12. Summary and Next Steps

In [ ]:
# ============================================================
# FINAL SUMMARY - Print summary of the fine-tuning process
# ============================================================

print("\n" + "=" * 60)
print("FINE-TUNING SUMMARY")
print("=" * 60)
print(f"\nModel: {MODEL_NAME}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nTest Set Performance:")
print(f"  F1 Micro: {test_results.get('eval_f1_micro', 0):.4f}")
print(f"  F1 Macro: {test_results.get('eval_f1_macro', 0):.4f}")
print(f"  Exact Match: {test_results.get('eval_exact_match', 0):.4f}")
print(f"  Hamming Loss: {test_results.get('eval_hamming_loss', 0):.4f}")
print(f"\nModel saved to: {OUTPUT_DIR}")
print("\n" + "=" * 60)
print("NEXT STEPS FOR PRODUCTION:")
print("=" * 60)
print("1. Copy the saved model to your backend server")
print("2. Load the model in your FastAPI application")
print("3. Create an API endpoint for predictions")
print("4. Test the endpoint with your React Native app")
print("\nExample FastAPI endpoint code:")
print("""
@app.post("/predict-interests")
async def predict_interests(text: str):
    result = predict_categories(text, model, tokenizer)
    return {"categories": result['predicted_categories']}
""")